In [4]:
# ============================================================
# NOTEBOOK : Nettoyage et Pré-traitement — GPS Spoofing
# Dataset  : Signal-level GPS attack captures
# ============================================================

# ────────────────────────────────────────
# 1. IMPORTATION DES LIBRAIRIES
# ────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# ────────────────────────────────────────
# 2. CHARGEMENT DES DATASETS
# ────────────────────────────────────────
PATH = r"C:\Drone_Attack_Similarity_Project\DATASET\GPS_Spoofing"

df1 = pd.read_csv(os.path.join(PATH, "GPS_Authentic.csv"))
df2 = pd.read_csv(os.path.join(PATH, "GPS_Simplified.csv"))
df3 = pd.read_csv(os.path.join(PATH, "GPS_Full.csv"))

df1_original = df1.copy()
df2_original = df2.copy()
df3_original = df3.copy()

print("  Fichiers chargés")
print(f"   df1 — GPS_Authentic  : {df1.shape[0]:,} lignes × {df1.shape[1]} colonnes")
print(f"   df2 — GPS_Simplified : {df2.shape[0]:,} lignes × {df2.shape[1]} colonnes")
print(f"   df3 — GPS_Full       : {df3.shape[0]:,} lignes × {df3.shape[1]} colonnes")

# ────────────────────────────────────────
# 3. SUPPRESSION DES DOUBLONS
# ────────────────────────────────────────
for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    nb = df.duplicated().sum()
    if nb > 0:
        df.drop_duplicates(inplace=True)
        print(f" {name} — {nb} doublons supprimés — nouveau shape : {df.shape}")
    else:
        print(f" {name} — Aucun doublon détecté")

# ────────────────────────────────────────
# 4. GESTION DES VALEURS MANQUANTES
# ────────────────────────────────────────
for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    total = df.isnull().sum().sum()
    print(f"\n {name} — Total valeurs manquantes : {total}")

    if total > 0:
        num_cols = df.select_dtypes(include=[np.number]).columns
        cat_cols = df.select_dtypes(include=['object']).columns

        for col in num_cols:
            if df[col].isnull().sum() > 0:
                median_val = df[col].median()
                df[col].fillna(median_val, inplace=True)
                print(f"  {col} — rempli par médiane ({median_val:.4f})")

        for col in cat_cols:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
                print(f"{col} — rempli par mode ({mode_val})")
    else:
        print(f" Aucune valeur manquante")

# ────────────────────────────────────────
# 5. SUPPRESSION DES COLONNES INUTILES
# ────────────────────────────────────────
for name, df_ref in [("df1", df1), ("df2", df2), ("df3", df3)]:
    unnamed_cols  = [c for c in df_ref.columns if 'Unnamed' in str(c)]
    zero_var_cols = [c for c in df_ref.columns
                     if df_ref[c].nunique() == 1 and c not in ['Output', 'label']]
    cols_to_drop  = list(set(unnamed_cols + zero_var_cols))

    if cols_to_drop:
        df_ref.drop(columns=cols_to_drop, inplace=True)
        print(f" {name} — {len(cols_to_drop)} colonnes supprimées")
    else:
        print(f" {name} — Aucune colonne inutile")
    print(f"   Shape après suppression : {df_ref.shape}")

# Vérification que df3 a bien des colonnes numériques
print(f"\n Colonnes numériques df3 : {df3.select_dtypes(include=[np.number]).columns.tolist()}")
print(f" Colonnes df3 : {df3.columns.tolist()}")

# ────────────────────────────────────────
# 6. TRAITEMENT DES OUTLIERS (IQR)
# ────────────────────────────────────────
num_cols_df2 = [c for c in df2.select_dtypes(include=[np.number]).columns
                if c not in ['Output', 'label_encoded']]
num_cols_df3 = [c for c in df3.select_dtypes(include=[np.number]).columns
                if c not in ['Output', 'label_encoded']]

print(f" num_cols_df2 ({len(num_cols_df2)}) : {num_cols_df2}")
print(f" num_cols_df3 ({len(num_cols_df3)}) : {num_cols_df3}")

nb_total_outliers = 0
for col in num_cols_df2:
    Q1 = df2[col].quantile(0.25)
    Q3 = df2[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    nb = ((df2[col] < lower) | (df2[col] > upper)).sum()
    nb_total_outliers += nb
    if nb > 0:
        df2[col] = df2[col].clip(lower=lower, upper=upper)
        print(f" df2 — {col} : {nb} outliers écrêtés")
print(f" Total outliers traités (df2) : {nb_total_outliers}")

nb_total_outliers_df3 = 0
for col in num_cols_df3:
    Q1 = df3[col].quantile(0.25)
    Q3 = df3[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    nb = ((df3[col] < lower) | (df3[col] > upper)).sum()
    nb_total_outliers_df3 += nb
    if nb > 0:
        df3[col] = df3[col].clip(lower=lower, upper=upper)
        print(f" df3 — {col} : {nb} outliers écrêtés")
print(f" Total outliers traités (df3) : {nb_total_outliers_df3}")

# ────────────────────────────────────────
# 7. NORMALISATION DES FEATURES
# ────────────────────────────────────────

# df1
num_cols_df1 = df1.select_dtypes(include=[np.number]).columns.tolist()
if num_cols_df1:
    scaler1 = StandardScaler()
    df1[num_cols_df1] = scaler1.fit_transform(df1[num_cols_df1])
    print(f" df1 — Normalisation appliquée sur {len(num_cols_df1)} features")
else:
    print(" df1 — Aucune colonne numérique à normaliser")

# df2
num_cols_df2 = [c for c in df2.select_dtypes(include=[np.number]).columns
                if c not in ['Output', 'label_encoded']]
if num_cols_df2:
    scaler2 = StandardScaler()
    df2[num_cols_df2] = scaler2.fit_transform(df2[num_cols_df2])
    print(f" df2 — Normalisation appliquée sur {len(num_cols_df2)} features")
else:
    print("  df2 — Aucune colonne numérique à normaliser")

# df3
num_cols_df3 = [c for c in df3.select_dtypes(include=[np.number]).columns
                if c not in ['Output', 'label_encoded']]
if num_cols_df3:
    scaler3 = StandardScaler()
    df3[num_cols_df3] = scaler3.fit_transform(df3[num_cols_df3])
    print(f"df3 — Normalisation appliquée sur {len(num_cols_df3)} features")
else:
    print(" df3 — Aucune colonne numérique à normaliser")
    print(f" Colonnes disponibles : {df3.columns.tolist()}")

# ────────────────────────────────────────
# 8. ENCODAGE DU LABEL (Output)
# ────────────────────────────────────────
# df1 n'a pas de label
# df2 et df3 ont une colonne 'Output'

le = LabelEncoder()

# df2
print(" df2 — Classes avant encodage :")
print(df2['Output'].value_counts())
df2['label_encoded'] = le.fit_transform(df2['Output'].astype(str))
print(f"\n df2 — Encodage appliqué :")
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
for classe, code in mapping.items():
    print(f"   {classe} → {code}")

le2 = LabelEncoder()

# df3
print(" df3 — Classes avant encodage :")
print(df3['Output'].value_counts())
df3['label_encoded'] = le2.fit_transform(df3['Output'].astype(str))
print(f"\n df3 — Encodage appliqué :")
mapping2 = dict(zip(le2.classes_, le2.transform(le2.classes_)))
for classe, code in mapping2.items():
    print(f"   {classe} → {code}")


# ────────────────────────────────────────
# 9. VÉRIFICATION FINALE
# ────────────────────────────────────────
print(" Vérification finale :\n")

for name, df_orig, df_clean in [
    ("df1", df1_original, df1),
    ("df2", df2_original, df2),
    ("df3", df3_original, df3)
]:
    print(f"   {name} :")
    print(f"      Shape original  : {df_orig.shape}")
    print(f"      Shape nettoyé   : {df_clean.shape}")
    print(f"      Valeurs manquantes : {df_clean.isnull().sum().sum()}")
    print(f"      Doublons restants  : {df_clean.duplicated().sum()}")
    print()


# ────────────────────────────────────────
# 10. SAUVEGARDE DES DATASETS NETTOYÉS
# ────────────────────────────────────────
OUTPUT_PATH = r"C:\Drone_Attack_Similarity_Project\Rapport\tables\GPS_Spoofing"

os.makedirs(OUTPUT_PATH, exist_ok=True)

df1.to_csv(os.path.join(OUTPUT_PATH, "GPS_Authentic_clean.csv"),  index=False)
df2.to_csv(os.path.join(OUTPUT_PATH, "GPS_Simplified_clean.csv"), index=False)
df3.to_csv(os.path.join(OUTPUT_PATH, "GPS_Full_clean.csv"),       index=False)

print("    Datasets nettoyés sauvegardés :")
print(f"   GPS_Authentic_clean.csv  : {df1.shape}")
print(f"   GPS_Simplified_clean.csv : {df2.shape}")
print(f"   GPS_Full_clean.csv       : {df3.shape}")
print(f"   Dossier : {OUTPUT_PATH}")

C:\Users\skhabaz\AppData\Local\Temp\ipykernel_17824\4154867177.py:21: DtypeWarning: Columns (0: PRN, 1: Unnamed: 1, 2: Unnamed: 2, 3: Unnamed: 3, 4: Unnamed: 4, 5: Unnamed: 5, 6: Unnamed: 6, 7: Unnamed: 7, 8: Carrier_Doppler_hz, 9: Unnamed: 9, 10: Unnamed: 10, 11: Unnamed: 11, 12: Unnamed: 12, 13: Unnamed: 13, 14: Unnamed: 14, 15: Unnamed: 15, 16: Pseudorange_m, 17: Unnamed: 17, 18: Unnamed: 18, 19: Unnamed: 19, 20: Unnamed: 20, 21: Unnamed: 21, 22: Unnamed: 22, 23: Unnamed: 23, 24: RX_time, 25: Unnamed: 25, 26: Unnamed: 26, 27: Unnamed: 27, 28: Unnamed: 28, 29: Unnamed: 29, 30: Unnamed: 30, 31: Unnamed: 31, 32: TOW_at_current_symbol_s, 33: Unnamed: 33, 34: Unnamed: 34, 35: Unnamed: 35, 36: Unnamed: 36, 37: Unnamed: 37, 38: Unnamed: 38, 39: Unnamed: 39, 40: Carrier_phase_cycles, 41: Unnamed: 41, 42: Unnamed: 42, 43: Unnamed: 43, 44: Unnamed: 44, 45: Unnamed: 45, 46: Unnamed: 46, 47: Unnamed: 47, 48: EC, 49: Unnamed: 49, 50: Unnamed: 50, 51: Unnamed: 51, 52: Unnamed: 52, 53: Unnamed: 53

  Fichiers chargés
   df1 — GPS_Authentic  : 159,146 lignes × 104 colonnes
   df2 — GPS_Simplified : 510,530 lignes × 14 colonnes
   df3 — GPS_Full       : 158,176 lignes × 112 colonnes
 df1 — 43396 doublons supprimés — nouveau shape : (115750, 104)
 df2 — 31218 doublons supprimés — nouveau shape : (479312, 14)
 df3 — 46862 doublons supprimés — nouveau shape : (111314, 112)

 df1 — Total valeurs manquantes : 0
 Aucune valeur manquante

 df2 — Total valeurs manquantes : 0
 Aucune valeur manquante

 df3 — Total valeurs manquantes : 0
 Aucune valeur manquante
 df1 — 91 colonnes supprimées
   Shape après suppression : (115750, 13)
 df2 — Aucune colonne inutile
   Shape après suppression : (479312, 14)
 df3 — 98 colonnes supprimées
   Shape après suppression : (111314, 14)

 Colonnes numériques df3 : []
 Colonnes df3 : ['PRN', 'Carrier_Doppler_hz', 'Pseudorange_m', 'RX_time', 'TOW_at_current_symbol_s', 'Carrier_phase_cycles', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0', 'Output']
 num_cols